# ExposoGraph

This notebook lets you build a carcinogen-gene knowledge graph using:

1. **LLM-assisted extraction** — describe a pathway in plain English and let GPT extract structured nodes and edges
2. **Manual entry** — add nodes and edges programmatically
3. **Load existing data** — import from a standalone interactive HTML export or a graph artifact

The graph can be **previewed interactively** inside this notebook and **exported** as a standalone interactive HTML file.


## 0. Install Dependencies (run once)

In [ ]:
%pip install networkx pydantic openai pyvis --quiet

## 1. Setup — Imports & Engine Initialization

In [ ]:
import json
import os
from pathlib import Path

from kg_builder import APP_NAME, APP_VERSION
from kg_builder.models import Node, Edge, NodeType, EdgeType, KnowledgeGraph
from kg_builder.engine import GraphEngine
from kg_builder.exporter import parse_graph_artifact, to_interactive_html, to_json
from kg_builder.llm_extractor import extract_graph, EXAMPLE_INPUT

engine = GraphEngine()
REFERENCE_VIEWER_DIR = Path("references/knowledge-graph")
DEPLOY_VIEWER_DIR = Path("knowledge-graph")
VIEWER_TEMPLATE_DIR = REFERENCE_VIEWER_DIR if REFERENCE_VIEWER_DIR.exists() else None


def find_graph_artifact() -> Path | None:
    candidates = [
        DEPLOY_VIEWER_DIR / "index.html",
        DEPLOY_VIEWER_DIR / "graph-data.js",
        REFERENCE_VIEWER_DIR / "graph-data.js",
        REFERENCE_VIEWER_DIR / "index.html",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


GRAPH_ARTIFACT = find_graph_artifact()
EXPORT_HTML_PATH = DEPLOY_VIEWER_DIR / "index.html"

print(f"{APP_NAME} v{APP_VERSION} notebook ready.")
if GRAPH_ARTIFACT is not None:
    print(f"Found graph artifact: {GRAPH_ARTIFACT}")
else:
    print("No existing graph artifact found. You can still build a graph manually or via LLM extraction.")
print(f"HTML export target: {EXPORT_HTML_PATH}")


---
## 2. Option A — LLM-Assisted Extraction

Describe a carcinogen metabolism pathway in plain English. The LLM will extract nodes (carcinogens, enzymes, metabolites, DNA adducts, pathways) and edges (ACTIVATES, DETOXIFIES, TRANSPORTS, FORMS_ADDUCT, REPAIRS, PATHWAY) automatically.

It can also capture manuscript-style annotations when present in the text, including `source_db`, `evidence`, `pmid`, `tissue`, `variant`, `phenotype`, and `activity_score`.

**Requirements:** Set `OPENAI_API_KEY` in your environment or enter it at the hidden prompt. Do not paste secrets into notebook cells. If no key is configured, the extraction cell will skip cleanly.


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
# Never paste a real API key into this notebook.
# Leave this as None to use OPENAI_API_KEY from your environment or a hidden prompt.
OPENAI_API_KEY = None
MODEL = "gpt-5.4-mini"        # alternatives: "gpt-4o-mini", "gpt-4.1", "gpt-4.1-mini", "gpt-4o"

def resolve_openai_api_key() -> str | None:
    configured_key = OPENAI_API_KEY or os.getenv("OPENAI_API_KEY")
    if configured_key:
        return configured_key

    from getpass import getpass

    entered_key = getpass("OpenAI API key (input hidden, press Enter to skip): ").strip()
    if entered_key:
        os.environ["OPENAI_API_KEY"] = entered_key
        return entered_key
    return None


In [ ]:
# ── Write your pathway description here ──────────────────────────────────
# Replace this with your own text, or use the built-in example.

user_text = """
Benzo[a]pyrene (BaP) is a Group 1 PAH carcinogen found in tobacco smoke.
CYP1A1 and CYP1B1 epoxidize BaP to BaP-7,8-epoxide (high reactivity).
EPHX1 hydrolyzes the epoxide to BaP-7,8-diol (intermediate reactivity).
A second epoxidation by CYP1A1 produces the ultimate carcinogen BPDE
(high reactivity), which forms BPDE-N2-dG DNA adducts repaired by
nucleotide excision repair enzymes XPC and ERCC2/XPD.
Detoxification is handled by GSTM1 and GSTP1, which conjugate BPDE
with glutathione to form BPDE-GSH (low reactivity), effluxed by ABCB1
and ABCC2. BaP also generates 8-oxo-dG via ROS, repaired by OGG1.
BaP maps to KEGG pathways 05204 (Chemical Carcinogenesis — DNA adducts)
and 00980 (Xenobiotic metabolism by CYP450).
"""

In [ ]:
# ── Run extraction ────────────────────────────────────────────────────────
resolved_key = resolve_openai_api_key()

if not resolved_key:
    kg = None
    print("Skipping extraction: no OpenAI API key configured.")
    print("Set OPENAI_API_KEY in your environment or re-run this cell and enter it at the hidden prompt.")
else:
    kg = extract_graph(user_text, model=MODEL, api_key=resolved_key)

    print(f"Extracted {len(kg.nodes)} nodes and {len(kg.edges)} edges\n")

    try:
        engine.merge(kg)
        print(f"Engine now has {engine.node_count} nodes, {engine.edge_count} edges")
    except ValueError as exc:
        print(f"Merge failed: {exc}")
        print("The extraction likely referenced a node that was not defined.")


In [ ]:
# ── Inspect what was extracted ─────────────────────────────────────────────
import pandas as pd

if kg is None:
    print("No extracted graph to inspect.")
else:
    node_df = pd.DataFrame([n.model_dump(mode="json") for n in kg.nodes])
    edge_df = pd.DataFrame([e.model_dump(mode="json") for e in kg.edges])

    print("── Nodes ──")
    display(node_df)

    print("\n── Edges ──")
    display(edge_df)


---
## 2. Option B — Manual Node & Edge Entry

Skip this section if you used LLM extraction above. You can also use both — manual entries merge additively into the graph.

In [ ]:
# ── Add nodes ─────────────────────────────────────────────────────────────
# Copy and modify these blocks to add your own nodes.

engine.add_node(Node(
    id="BaP",
    label="Benzo[a]pyrene",
    type=NodeType.CARCINOGEN,
    detail="Prototypical PAH; tobacco smoke, grilled food, coal tar",
    group="PAH",
    iarc="Group 1",
    source_db="IARC; KEGG",
    evidence="Index PAH carcinogen used in the manuscript knowledge graph",
))

engine.add_node(Node(
    id="CYP1A1",
    label="CYP1A1",
    type=NodeType.ENZYME,
    detail="PAH → diol-epoxides; extrahepatic expression; AhR-inducible",
    phase="I",
    role="Activation",
    source_db="KEGG; PharmVar; GTEx",
    tissue="lung",
    variant="CYP1A1*2C",
    phenotype="increased activity",
))

engine.add_node(Node(
    id="BPDE",
    label="BPDE",
    type=NodeType.METABOLITE,
    detail="Benzo[a]pyrene-7,8-diol-9,10-epoxide; ultimate PAH carcinogen",
    reactivity="High",
    source_db="KEGG",
))

engine.add_node(Node(
    id="BPDE_dG",
    label="BPDE-N2-dG",
    type=NodeType.DNA_ADDUCT,
    detail="Major PAH-DNA adduct; G→T transversions in p53 codon 249",
    source_db="IARC; CTD",
))

engine.add_node(Node(
    id="GSTM1",
    label="GSTM1",
    type=NodeType.ENZYME,
    detail="Glutathione conjugation of PAH diol-epoxides, AFB1-epoxide",
    phase="II",
    role="Detoxification",
    source_db="PharmVar; CPIC",
    variant="GSTM1-null",
    phenotype="loss of function",
))

engine.add_node(Node(
    id="XPC",
    label="XPC",
    type=NodeType.ENZYME,
    detail="Nucleotide excision repair; bulky adduct recognition",
    phase="Repair",
    role="Repair",
    source_db="KEGG; CTD",
))

print(f"Nodes in graph: {engine.node_count}")

In [ ]:
# ── Add edges ─────────────────────────────────────────────────────────────
# Copy and modify these blocks to add your own edges.

engine.add_edge(Edge(
    source="CYP1A1", target="BPDE",
    type=EdgeType.ACTIVATES,
    label="epoxidation",
    carcinogen="BaP",
    source_db="KEGG",
    tissue="lung",
    evidence="Phase I activation of PAH to an ultimate carcinogen",
))

engine.add_edge(Edge(
    source="BPDE", target="BPDE_dG",
    type=EdgeType.FORMS_ADDUCT,
    label="N2-dG adduct",
    source_db="CTD; IARC",
))

engine.add_edge(Edge(
    source="GSTM1", target="BPDE",
    type=EdgeType.DETOXIFIES,
    label="GSH conjugation",
    carcinogen="BaP",
    source_db="KEGG; CPIC",
    evidence="Loss of GSTM1 activity can reduce detoxification capacity",
))

engine.add_edge(Edge(
    source="XPC", target="BPDE_dG",
    type=EdgeType.REPAIRS,
    label="NER recognition",
    source_db="KEGG",
))

print(f"Edges in graph: {engine.edge_count}")

---
## 2. Option C — Load Existing Graph Data

Import the data from an existing graph artifact. The notebook will look for these in order:

1. `knowledge-graph/index.html`
2. `knowledge-graph/graph-data.js`
3. `references/knowledge-graph/graph-data.js`
4. `references/knowledge-graph/index.html`


In [ ]:
# ── Load from an existing graph artifact ──────────────────────────────────
artifact_path = GRAPH_ARTIFACT or find_graph_artifact()

if artifact_path is not None:
    kg = parse_graph_artifact(artifact_path)
    engine.load(kg)
    GRAPH_ARTIFACT = artifact_path
    print(f"Loaded {len(kg.nodes)} nodes and {len(kg.edges)} edges from {artifact_path}")
else:
    print("No graph artifact found in knowledge-graph/ or references/knowledge-graph/.")
    print("Skip this cell or build a graph another way.")

print(f"\nEngine totals: {engine.node_count} nodes, {engine.edge_count} edges")


In [ ]:
# ── Or load from a JSON file ──────────────────────────────────────────────
# Uncomment to load a previously-exported JSON:

# with open("knowledge_graph.json") as f:
#     data = json.load(f)
# kg = KnowledgeGraph(**data)
# engine.load(kg)
# print(f"Loaded {len(kg.nodes)} nodes and {len(kg.edges)} edges from JSON")

---
## 3. Validate & Inspect

In [ ]:
import pandas as pd

# ── Summary stats ─────────────────────────────────────────────────────────
print(f"Total nodes: {engine.node_count}")
print(f"Total edges: {engine.edge_count}\n")

# Node type breakdown
node_data = [dict(d) for _, d in engine.G.nodes(data=True)]
if node_data:
    ndf = pd.DataFrame(node_data)
    print("── Node type counts ──")
    display(ndf["type"].value_counts().to_frame("count"))

    anno_cols = [c for c in ["source_db", "pmid", "tissue", "variant", "phenotype", "activity_score"] if c in ndf.columns]
    if anno_cols:
        print("\n── Node annotations ──")
        display(ndf[["id", "label", "type", *anno_cols]].fillna(""))

# Edge type breakdown
edge_data = [dict(d) for _, _, _, d in engine.G.edges(keys=True, data=True)]
if edge_data:
    edf = pd.DataFrame(edge_data)
    print("\n── Edge type counts ──")
    display(edf["type"].value_counts().to_frame("count"))

    edge_anno_cols = [c for c in ["source_db", "pmid", "tissue", "evidence"] if c in edf.columns]
    if edge_anno_cols:
        print("\n── Edge annotations ──")
        display(edf[["source", "target", "type", *edge_anno_cols]].fillna(""))

In [ ]:
# ── Validation ────────────────────────────────────────────────────────────
errors = engine.validate()
if errors:
    print(f"⚠ {len(errors)} validation issue(s):")
    for e in errors:
        print(f"  - {e}")
else:
    print("✓ Graph is valid — no dangling edges or missing references.")

---
## 4. Interactive Graph Preview

Builds an interactive, zoomable, draggable graph preview using [pyvis](https://pyvis.readthedocs.io/).

Note: VS Code notebooks can blank the entire webview when a full PyVis HTML document is embedded inline. This section therefore saves a local HTML preview file instead of rendering the full document inside the notebook.

In [ ]:
from pyvis.network import Network
from IPython.display import Markdown, display as ipy_display

NODE_COLORS = {
    "Carcinogen": "#e05565",
    "Enzyme": "#4f98a3",
    "Gene": "#3d8b8b",
    "Metabolite": "#e8945a",
    "DNA_Adduct": "#a86fdf",
    "Pathway": "#5591c7",
    "Tissue": "#c2855a",
}
NODE_SIZES = {
    "Carcinogen": 30,
    "Enzyme": 22,
    "Gene": 22,
    "Metabolite": 18,
    "DNA_Adduct": 20,
    "Pathway": 25,
    "Tissue": 18,
}
NODE_SHAPES = {
    "Carcinogen": "diamond",
    "Enzyme": "dot",
    "Gene": "dot",
    "Metabolite": "dot",
    "DNA_Adduct": "hexagon",
    "Pathway": "box",
    "Tissue": "ellipse",
}
EDGE_COLORS = {
    "ACTIVATES": "#e05565",
    "DETOXIFIES": "#6daa45",
    "TRANSPORTS": "#5591c7",
    "FORMS_ADDUCT": "#a86fdf",
    "REPAIRS": "#e8af34",
    "PATHWAY": "#555b6a",
    "EXPRESSED_IN": "#c2855a",
    "INDUCES": "#d4a843",
    "INHIBITS": "#8b4a6b",
    "ENCODES": "#3d8b8b",
}


def annotation_lines(data: dict) -> list[str]:
    lines = []
    for label, key in [
        ("Source", "source_db"),
        ("Evidence", "evidence"),
        ("PMID", "pmid"),
        ("Tissue", "tissue"),
        ("Variant", "variant"),
        ("Phenotype", "phenotype"),
    ]:
        if data.get(key):
            lines.append(f"<br><b>{label}:</b> {data[key]}")
    if data.get("activity_score") is not None:
        lines.append(f"<br><b>Activity score:</b> {data['activity_score']}")
    return lines


def show_graph(eng: GraphEngine, height: str = "700px", out_path: str | Path = "kg_preview.html") -> Path:
    """Save the knowledge graph as an interactive pyvis HTML preview."""
    net = Network(
        height=height,
        width="100%",
        directed=True,
        notebook=True,
        bgcolor="#0f1117",
        font_color="#e1e4ea",
        cdn_resources="remote",
    )

    for node_id, data in eng.G.nodes(data=True):
        ntype = data.get("type", "Enzyme")
        net.add_node(
            node_id,
            label=data.get("label", node_id),
            color=NODE_COLORS.get(ntype, "#888"),
            size=NODE_SIZES.get(ntype, 20),
            shape=NODE_SHAPES.get(ntype, "dot"),
            title=(
                f"<b>{data.get('label', node_id)}</b><br>"
                f"<i>{ntype}</i><br>"
                f"{data.get('detail', '')}"
                + "".join(annotation_lines(data))
            ),
        )

    for u, v, _, data in eng.G.edges(keys=True, data=True):
        etype = data.get("type", "ACTIVATES")
        net.add_edge(
            u, v,
            title=(data.get("label", etype) + "".join(annotation_lines(data))),
            color=EDGE_COLORS.get(etype, "#555"),
            width=2,
            arrows="to",
        )

    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -80,
          "centralGravity": 0.008,
          "springLength": 150,
          "springConstant": 0.04,
          "damping": 0.5
        },
        "solver": "forceAtlas2Based",
        "stabilization": { "iterations": 200 }
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 100,
        "navigationButtons": true
      }
    }
    """)

    out_path = Path(out_path)
    net.save_graph(str(out_path))
    ipy_display(Markdown(
        f"Interactive preview saved to `{out_path}`.  \n"
        f"Open it in a browser for the full graph view.  \n"
        f"Inline rendering is intentionally disabled here to avoid VS Code notebook renderer crashes."
    ))
    return out_path


preview_path = show_graph(engine)
print(f"Saved preview to {preview_path.resolve()}")


---
## 5. Export

Export the graph as a standalone interactive HTML file. A JSON snapshot is also available as an optional raw-data export.


In [ ]:
# ── Export to a standalone interactive HTML file ─────────────────────────
EXPORT_HTML_PATH.parent.mkdir(parents=True, exist_ok=True)
out = to_interactive_html(engine, EXPORT_HTML_PATH, template_path=VIEWER_TEMPLATE_DIR)
GRAPH_ARTIFACT = out
print(f"✓ Exported standalone interactive HTML to {out}")
print("  Open this file in a browser to explore the graph.")


In [ ]:
# ── Optional: export a standalone JSON snapshot ──────────────────────────
out = to_json(engine, "knowledge_graph.json")
print(f"✓ Exported JSON snapshot to {out}")


---
## 6. Add More Pathways (Iterative Use)

You can keep running the cells below to add more carcinogen pathways to the same graph. Each extraction or manual addition **merges** into the existing graph immediately.

That means there is nothing extra to "add back" into Section 3. Once `engine.merge(...)` runs here, the live graph is updated and Section 7 will show that updated state.


In [ ]:
additional_text = """
Aflatoxin B1 (AFB1), produced by Aspergillus molds, is bioactivated in the liver by CYP1A2 and CYP3A4 to the reactive AFB1-8,9-epoxide.
The epoxide forms a covalent DNA adduct with guanine, creating AFB1-N7-Gua, which can lead to mutagenesis.
Glutathione S-transferases such as GSTM1 can detoxify the epoxide via glutathione conjugation.
AFB1 maps to KEGG 05204 (Chemical Carcinogenesis — DNA adducts).
"""


def edge_identity(edge) -> str:
    return f"{edge.source}-{edge.type.value}-{edge.target}"


RUN_ADDITIONAL_EXTRACTION = False
resolved_key = resolve_openai_api_key()

if not RUN_ADDITIONAL_EXTRACTION:
    print("Set RUN_ADDITIONAL_EXTRACTION = True to extract and merge this pathway.")
elif not resolved_key:
    print("No OpenAI API key configured. Set OPENAI_API_KEY in your environment or re-run and enter it at the hidden prompt.")
else:
    before_node_ids = set(engine.G.nodes)
    before_edge_keys = {key for _, _, key in engine.G.edges(keys=True)}

    kg2 = extract_graph(additional_text, model=MODEL, api_key=resolved_key)

    extracted_node_ids = [node.id for node in kg2.nodes]
    extracted_edge_keys = [edge_identity(edge) for edge in kg2.edges]
    overlapping_nodes = sorted(node_id for node_id in extracted_node_ids if node_id in before_node_ids)
    overlapping_edges = sorted(edge_key for edge_key in extracted_edge_keys if edge_key in before_edge_keys)

    print(f"Second extraction produced {len(kg2.nodes)} nodes and {len(kg2.edges)} edges.")
    print(f"Overlapping nodes: {overlapping_nodes or "none"}")
    print(f"Overlapping edges: {overlapping_edges or "none"}")

    try:
        engine.merge(kg2)
        print(f"Merged graph now has {engine.node_count} nodes and {engine.edge_count} edges.")
    except ValueError as exc:
        print(f"Merge failed: {exc}")


### 6A. Compare The Latest Extraction Against The Current Graph

Run this after Section 6. It shows which extracted nodes and edges were new versus already present, so you can compare prompt 1 and prompt 2 directly.


In [ ]:
if "latest_extraction_nodes_df" not in globals() or "latest_extraction_edges_df" not in globals():
    print("No additional extraction has been run in Section 6 yet.")
else:
    print("Latest extracted nodes:")
    display(latest_extraction_nodes_df)

    print("\nLatest extracted edges:")
    display(latest_extraction_edges_df)

    if not stashed_added_nodes.empty:
        print("\nNet new node IDs added to the live graph:")
        display(stashed_added_nodes)
    else:
        print("\nNo net new node IDs were added.")

    if not stashed_added_edges.empty:
        print("\nNet new edge identities added to the live graph:")
        display(stashed_added_edges)
    else:
        print("\nNo net new edge identities were added.")


---
## 7. Utilities

In [ ]:
# ── Browse all nodes ──────────────────────────────────────────────────────
all_nodes = pd.DataFrame([dict(d) for _, d in engine.G.nodes(data=True)])
if not all_nodes.empty:
    display(all_nodes.sort_values("type"))
else:
    print("Graph is empty.")

In [ ]:
# ── Browse all edges ──────────────────────────────────────────────────────
all_edges = pd.DataFrame([dict(d) for _, _, d in engine.G.edges(data=True)])
if not all_edges.empty:
    display(all_edges.sort_values("type"))
else:
    print("No edges yet.")

In [ ]:
# ── Delete a node (and its connected edges) ──────────────────────────────
# engine.remove_node("BaP")

# ── Clear the entire graph ───────────────────────────────────────────────
# engine.clear()

# ── Check current state ──────────────────────────────────────────────────
print(f"Graph: {engine.node_count} nodes, {engine.edge_count} edges")

---
## 7A. Validate Current Graph Before Export

Run this section after Section 7 to confirm that the current live graph is the one you want to export.

This is the review checkpoint for changes from Section 6 and any manual edits. If the counts or validation output are not what you expect, fix the graph before running the updated export step.


In [ ]:
print(f"Current graph: {engine.node_count} nodes, {engine.edge_count} edges")

errors = engine.validate()
if errors:
    print(f"\nValidation issues: {len(errors)}")
    for error in errors:
        print(f"  - {error}")
else:
    print("\nValidation passed.")

node_df = pd.DataFrame([dict(d) for _, d in engine.G.nodes(data=True)])
edge_df = pd.DataFrame([dict(d) for _, _, _, d in engine.G.edges(keys=True, data=True)])

if not node_df.empty:
    print("\nNode type counts:")
    display(node_df["type"].value_counts().to_frame("count"))

if not edge_df.empty:
    print("\nEdge type counts:")
    display(edge_df["type"].value_counts().to_frame("count"))


---
## 7B. Refresh Preview & Export Updated HTML

After using the utilities above, run this section to regenerate the local preview and write a new standalone HTML export with the `_updated.html` suffix.


In [ ]:
UPDATED_PREVIEW_PATH = Path("kg_preview_updated.html")
UPDATED_EXPORT_PATH = EXPORT_HTML_PATH.with_name(f"{EXPORT_HTML_PATH.stem}_updated{EXPORT_HTML_PATH.suffix}")

updated_preview = show_graph(engine, out_path=UPDATED_PREVIEW_PATH)
UPDATED_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
updated_export = to_interactive_html(engine, UPDATED_EXPORT_PATH, template_path=VIEWER_TEMPLATE_DIR)
GRAPH_ARTIFACT = updated_export

print(f"Updated preview: {updated_preview.resolve()}")
print(f"Updated standalone export: {updated_export.resolve()}")


---
## 8. Manuscript Figures Review

These manuscript figures are closely related to the knowledge graph, but they are not all the same kind of artifact.

- `fig1_architecture.pdf` is a conceptual architecture summary derived from the graph schema and manuscript counts.
- `fig2_graph_overview.pdf` appears to be the closest to a direct full-network knowledge-graph figure.
- `fig3_exemplar_pathways.pdf` and `fig4_androgen_dual.pdf` are pathway-focused figures built from the same biological model, but they also include additional pharmacogenomic and mechanistic annotations beyond the current graph artifact.

Notable mismatches with the current graph data:

- `fig1` references `MUTYH`, which is not present in the current graph.
- `fig3` includes `AFB1-FAPY` and `Apurinic site`, which are not present in the current graph.
- `fig4` includes `SULT1E1`, which is not present in the current graph.

So the figures are best understood as **knowledge-graph-derived manuscript figures**, not all as literal exports of the exact current graph artifact.


In [ ]:
import pandas as pd
from pathlib import Path

figure_review = pd.DataFrame([
    {
        "figure": "fig1_architecture.pdf",
        "assessment": "Schema-derived",
        "kg_relationship": "Summarizes node/edge ontology and graph counts",
        "notes": "Conceptually derived from the KG, but includes MUTYH in the repair set, which is absent from the current graph artifact.",
    },
    {
        "figure": "fig2_graph_overview.pdf",
        "assessment": "Direct KG overview",
        "kg_relationship": "Represents the full network overview",
        "notes": "Most likely the closest figure to a direct export or screenshot of the current knowledge graph.",
    },
    {
        "figure": "fig3_exemplar_pathways.pdf",
        "assessment": "KG-derived, enriched",
        "kg_relationship": "Uses graph pathways as exemplars",
        "notes": "Adds pharmacogenomic variants and extra pathway elements like AFB1-FAPY, apurinic site, and MUTYH beyond the current graph artifact.",
    },
    {
        "figure": "fig4_androgen_dual.pdf",
        "assessment": "KG-derived, enriched",
        "kg_relationship": "Expands the androgen subgraph into a manuscript mechanism figure",
        "notes": "Aligned with androgen dual-mechanism concept in the graph, but includes SULT1E1 and other detail beyond the current graph artifact.",
    },
])

display(figure_review)


In [ ]:
from IPython.display import IFrame, display, Markdown

figure_paths = [
    Path("fig1_architecture.pdf"),
    Path("fig2_graph_overview.pdf"),
    Path("fig3_exemplar_pathways.pdf"),
    Path("fig4_androgen_dual.pdf"),
]

for fig_path in figure_paths:
    if fig_path.exists():
        display(Markdown(f"### {fig_path.name}"))
        display(IFrame(src=str(fig_path), width="100%", height=700))
    else:
        display(Markdown(f"### {fig_path.name}"))
        print(f"File not found: {fig_path}")

In [ ]:
# Optional: open a single figure by name
# selected = Path("fig2_graph_overview.pdf")
# display(IFrame(src=str(selected), width="100%", height=800))

### Figure Consistency Check

This section compares figure-specific entities against the canonical graph artifact currently available in the project.

It reports three cases:

- `exact` — the figure entity exists directly in the graph
- `alias` — the figure label appears to correspond to an existing graph node with a slightly different name
- `missing` — the figure entity does not exist in the current graph and is likely a manuscript-only extension


In [ ]:
from pathlib import Path
import pandas as pd

artifact_path = GRAPH_ARTIFACT or find_graph_artifact()

if artifact_path is None:
    print("No graph artifact available for the consistency check.")
else:
    canonical_kg = parse_graph_artifact(artifact_path)
    canonical_nodes = canonical_kg.nodes

    node_labels = {n.label for n in canonical_nodes}
    node_ids = {n.id for n in canonical_nodes}
    label_to_id = {n.label: n.id for n in canonical_nodes}

    # Curated figure mentions that are relevant to manuscript-vs-graph consistency.
    figure_entities = {
        "fig1_architecture.pdf": [
            "MUTYH",
        ],
        "fig3_exemplar_pathways.pdf": [
            "BaP-7,8-oxide",
            "BPDE-dG Adduct",
            "dG-ABP Adduct",
            "5-DHT",
            "17-Estradiol",
            "CE Quinones",
            "DNA Adducts",
            "AFB1-N7-Gua Adduct",
            "Apurinic site",
            "AFB1-FAPY",
            "MUTYH",
        ],
        "fig4_androgen_dual.pdf": [
            "5-DHT",
            "17-Estradiol",
            "4-OH-Estradiol",
            "CE-3,4-Quinones",
            "DNA ADDUCTS",
            "6-OH-T",
            "SULT1E1",
        ],
    }

    # Manual alias mapping for figure labels that clearly correspond to existing graph nodes.
    alias_map = {
        "BaP-7,8-oxide": "BaP-7,8-epoxide",
        "BPDE-dG Adduct": "BPDE-N2-dG",
        "dG-ABP Adduct": "4-ABP-C8-dG",
        "5-DHT": "5α-DHT",
        "17-Estradiol": "17β-Estradiol",
        "4-OH-Estradiol": "4-OH-Estradiol",
        "CE Quinones": "E2-3,4-Quinone",
        "CE-3,4-Quinones": "E2-3,4-Quinone",
        "DNA Adducts": "E2-depurinating",
        "DNA ADDUCTS": "E2-depurinating",
        "AFB1-N7-Gua Adduct": "AFB1-N7-dG",
        "6-OH-T": "6β-OH-Testosterone",
    }

    rows = []
    for figure, entities in figure_entities.items():
        for entity in entities:
            if entity in node_labels or entity in node_ids:
                rows.append({
                    "figure": figure,
                    "figure_entity": entity,
                    "status": "exact",
                    "mapped_graph_node": entity,
                })
            elif entity in alias_map and alias_map[entity] in node_labels:
                rows.append({
                    "figure": figure,
                    "figure_entity": entity,
                    "status": "alias",
                    "mapped_graph_node": alias_map[entity],
                })
            else:
                rows.append({
                    "figure": figure,
                    "figure_entity": entity,
                    "status": "missing",
                    "mapped_graph_node": "",
                })

    consistency_df = pd.DataFrame(rows)
    display(consistency_df)

    print(f"\nCanonical graph artifact: {artifact_path}")
    print("\nSummary by figure:")
    display(consistency_df.groupby(["figure", "status"]).size().unstack(fill_value=0))

    missing_df = consistency_df[consistency_df["status"] == "missing"]
    if not missing_df.empty:
        print("\nEntities mentioned in figures but not present in the current graph artifact:")
        display(missing_df[["figure", "figure_entity"]])
    else:
        print("\nAll reviewed figure entities map to the current graph.")
